### Imports & Schema Definition

This section defines all required imports and the Bronze schema.  
The schema ensures consistent typing for market cap, sector, industry, and ingestion timestamps when writing to the Delta table.

In [1]:
# Imports
import yfinance as yf
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

# Bronze schema definition
schema = StructType([
    StructField("cik", StringType(), True),
    StructField("ticker", StringType(), True),
    StructField("company_name", StringType(), True),
    StructField("market_cap", DoubleType(), True),
    StructField("sector", StringType(), True),
    StructField("industry", StringType(), True),
    StructField("ingestion_ts", TimestampType(), True)
])


StatementMeta(, baf7c425-5408-47a8-96c8-1b576ff5bdc9, 5, Finished, Available, Finished, False)

### Load dimension table & determine companies to process

We load the dimension table containing all companies and tickers which have a DEF14A document.
If the Bronze table already exists, we only process companies that are not yet stored.
This ensures incremental ingestion and avoids duplicate writes.

In [2]:
# Check if Bronze table exists
bronze_exists = False
try:
    spark.table("bronze_company_info")
    bronze_exists = True
except:
    bronze_exists = False

# Load ompanies to process
comp_list = spark.table("sec_def14a_bronze")
dim = (
    comp_list
    .select("cik", "ticker", "company_name")
    .distinct()
)


# Determine which companies still need processing
if bronze_exists:
    bronze = spark.table("bronze_company_info").select("cik")
    new_companies = dim.join(bronze, "cik", "left_anti")
    print(f"Existing Bronze table found. New companies to process: {new_companies.count()}")
else:
    new_companies = dim
    print(f"No Bronze table found. Processing ALL companies: {new_companies.count()}")


StatementMeta(, baf7c425-5408-47a8-96c8-1b576ff5bdc9, 6, Finished, Available, Finished, False)

Existing Bronze table found. New companies to process: 267


### Convert to Python

Spark cannot parallelize external API calls.  
Therefore, we collect the companies into a Python list so they can be processed using a thread pool.

In [3]:
# Convert to list for parallel processing
companies = [(r["cik"], r["ticker"], r["company_name"]) for r in new_companies.collect()]
# Limit to 500 companies per spark session to avoid Yahoo Finance blocking
companies = companies[:500]
total = len(companies)


StatementMeta(, baf7c425-5408-47a8-96c8-1b576ff5bdc9, 7, Finished, Available, Finished, False)

### Define the ticker ingestion function

This function fetches market cap, sector, and industry for a single ticker.  
Errors are handled gracefully so that individual failures do not interrupt the overall ingestion.


In [4]:
# Ticker ingestion function
def fetch_info(cik, ticker, company_name):
    try:
        t = yf.Ticker(ticker)
        fast = t.fast_info
        info = t.info

        market_cap_raw = fast.get("market_cap") or info.get("marketCap")
        market_cap = float(market_cap_raw) if market_cap_raw is not None else None

        return Row(
            cik=str(cik),
            ticker=str(ticker),
            company_name=str(company_name),   
            market_cap=market_cap,
            sector=info.get("sector"),
            industry=info.get("industry"),
            ingestion_ts=datetime.utcnow()
        )
    except Exception:
        return Row(
            cik=str(cik),
            ticker=str(ticker),
            company_name=str(company_name),  
            market_cap=None,
            sector=None,
            industry=None,
            ingestion_ts=datetime.utcnow()
        )


StatementMeta(, baf7c425-5408-47a8-96c8-1b576ff5bdc9, 8, Finished, Available, Finished, False)

### Parallel ingestion using ThreadPoolExecutor

This section parallelizes the ingestion of all tickers using a thread pool.  
API calls are I/O‑bound, meaning they spend most of their time waiting for network responses.
Python can overlap these waiting periods across multiple threads, making the ingestion significantly faster.
Progress logs provide visibility during long runs.

In [5]:
# Parallel execution
rows = []
start_time = time.time()

with ThreadPoolExecutor(max_workers=20) as executor:
    futures = {
        executor.submit(fetch_info, cik, ticker, company_name): (cik, ticker)
        for cik, ticker, company_name in companies
    }

    for idx, future in enumerate(as_completed(futures), start=1):
        cik, ticker = futures[future]
        elapsed = (time.time() - start_time) / 60
        print(f"[{idx}/{total}] {ticker} - {idx/total*100:.2f}% - elapsed {elapsed:.2f} min")

        rows.append(future.result())


StatementMeta(, baf7c425-5408-47a8-96c8-1b576ff5bdc9, 9, Finished, Available, Finished, False)

[1/267] LPTH - 0.37% - elapsed 0.01 min
[2/267] FATE - 0.75% - elapsed 0.01 min
[3/267] PBYI - 1.12% - elapsed 0.01 min
[4/267] FNKO - 1.50% - elapsed 0.01 min
[5/267] AAME - 1.87% - elapsed 0.01 min
[6/267] ORLY - 2.25% - elapsed 0.01 min
[7/267] IOSP - 2.62% - elapsed 0.01 min
[8/267] MCRB - 3.00% - elapsed 0.01 min
[9/267] BHB - 3.37% - elapsed 0.01 min
[10/267] MS - 3.75% - elapsed 0.01 min
[11/267] FSS - 4.12% - elapsed 0.02 min
[12/267] HFFG - 4.49% - elapsed 0.02 min
[13/267] RSRV - 4.87% - elapsed 0.02 min
[14/267] JBI - 5.24% - elapsed 0.02 min
[15/267] TEAM - 5.62% - elapsed 0.02 min
[16/267] HBAN - 5.99% - elapsed 0.02 min
[17/267] DBD - 6.37% - elapsed 0.02 min
[18/267] POWL - 6.74% - elapsed 0.02 min
[19/267] FICO - 7.12% - elapsed 0.02 min
[20/267] AIRG - 7.49% - elapsed 0.02 min
[21/267] SSD - 7.87% - elapsed 0.02 min
[22/267] SMP - 8.24% - elapsed 0.02 min
[23/267] PROV - 8.61% - elapsed 0.02 min
[24/267] SSTI - 8.99% - elapsed 0.02 min
[25/267] NRIM - 9.36% - elapsed 0

### Write results to Bronze table

Finally, we write all collected rows into the Bronze Delta table.  
If the table exists, new rows are appended.
If not, the table is created with the initial dataset.

In [6]:
# Write to Bronze
if len(rows) == 0:
    print("No new companies to process. Nothing to write.")
else:
    df = spark.createDataFrame(rows, schema=schema)

    if bronze_exists:
        df.write.format("delta").mode("append").saveAsTable("bronze_company_info")
        print("Appended new companies to bronze_company_info.")
    else:
        df.write.format("delta").mode("overwrite").saveAsTable("bronze_company_info")
        print("Created bronze_company_info with initial data.")


StatementMeta(, baf7c425-5408-47a8-96c8-1b576ff5bdc9, 10, Finished, Available, Finished, False)

Appended new companies to bronze_company_info.
